# 🌍 Profil de risque national & Adaptation — Multi-péril (CLIMADA)

Agrégation multi-aléa pour construire des profils de risque nationaux et évaluer des stratégies d'adaptation via le moteur CostBenefit de CLIMADA.

**Approche** : Multi-péril → profil national → coût-bénéfice adaptation  
**Zone** : France, Jamaïque, Bangladesh  
**Exposition** : LitPop + BlackMarble  
**Moteur** : CostBenefit, UQ Engine

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from climada.hazard import Hazard, Centroids
from climada.entity import Exposures, ImpactFuncSet, ImpactFunc, MeasureSet, Measure
from climada.engine import Impact

try:
    from climada.engine import CostBenefit
    CB_OK = True
except ImportError:
    CB_OK = False
    print("⚠️ CostBenefit non disponible")

print(f"CostBenefit disponible : {CB_OK}")

## 1. Profils de risque nationaux

### Approche multi-péril
Pour chaque pays, on construit un profil en agrégeant les impacts de plusieurs aléas :
- **France** : Tempête, inondation, sécheresse, feux
- **Jamaïque** : Cyclone tropical (vent, pluie, surge), inondation côtière
- **Bangladesh** : Cyclone tropical, inondation fluviale, submersion côtière

### Exposition nationale
CLIMADA propose deux sources d'exposition nationale :
- **LitPop** : Nightlights × Population → proxy de la valeur des actifs
- **BlackMarble** : NASA VIIRS nightlights (résolution ~500m)

In [ ]:
# --- LitPop réel (nécessite téléchargement) ---
# from climada.entity import LitPop
# exposure_fra = LitPop.from_countries(countries=['FRA'], res_arcsec=150, fin_mode='gdp')
# exposure_jam = LitPop.from_countries(countries=['JAM'], res_arcsec=150, fin_mode='gdp')
# exposure_bgd = LitPop.from_countries(countries=['BGD'], res_arcsec=150, fin_mode='gdp')

# --- Exposition synthétique par pays ---
from shapely.geometry import Point
import geopandas as gpd
from scipy.sparse import csr_matrix, vstack

np.random.seed(42)

def create_synthetic_exposure(country, cities, n_assets, total_gdp):
    """Créer une exposition synthétique pour un pays."""
    lons, lats, values = [], [], []
    
    for city, (clon, clat, weight) in cities.items():
        if city.startswith('_'):
            continue
        n = max(1, int(n_assets * weight))
        lons.extend(np.random.normal(clon, 0.3, n))
        lats.extend(np.random.normal(clat, 0.2, n))
        values.extend(np.random.exponential(total_gdp * weight / n, n))
    
    remaining = n_assets - len(lons)
    bbox = cities.get('_bbox', (-5, 42, 10, 51))
    if remaining > 0:
        lons.extend(np.random.uniform(bbox[0], bbox[2], remaining))
        lats.extend(np.random.uniform(bbox[1], bbox[3], remaining))
        values.extend(np.random.exponential(total_gdp / n_assets * 0.5, remaining))
    
    gdf = gpd.GeoDataFrame({
        'value': np.array(values[:n_assets]),
        'geometry': [Point(lo, la) for lo, la in zip(lons[:n_assets], lats[:n_assets])]
    }, crs='EPSG:4326')
    
    return gdf

# France
france_cities = {
    'Paris': (2.35, 48.86, 0.30),
    'Lyon': (4.83, 45.76, 0.08),
    'Marseille': (5.37, 43.30, 0.07),
    'Toulouse': (1.44, 43.60, 0.05),
    'Bordeaux': (-0.58, 44.84, 0.05),
    'Lille': (3.06, 50.63, 0.05),
    'Nantes': (-1.55, 47.22, 0.04),
    'Strasbourg': (7.75, 48.57, 0.04),
    'Nice': (7.26, 43.71, 0.04),
    'Rennes': (-1.68, 48.11, 0.03),
    '_bbox': (-5, 42, 10, 51),
}

jamaica_cities = {
    'Kingston': (-76.79, 18.0, 0.45),
    'Montego_Bay': (-77.92, 18.47, 0.15),
    'Spanish_Town': (-76.95, 18.0, 0.10),
    'Portmore': (-76.88, 17.95, 0.08),
    'Mandeville': (-77.50, 18.04, 0.07),
    '_bbox': (-78.4, 17.7, -76.2, 18.6),
}

bangladesh_cities = {
    'Dhaka': (90.41, 23.81, 0.35),
    'Chittagong': (91.83, 22.36, 0.12),
    'Khulna': (89.56, 22.84, 0.06),
    'Rajshahi': (88.60, 24.37, 0.05),
    'Sylhet': (91.87, 24.90, 0.04),
    'Comilla': (91.20, 23.46, 0.04),
    'Barisal': (90.37, 22.70, 0.04),
    '_bbox': (88.0, 20.5, 92.7, 26.6),
}

countries = {
    'France': {'cities': france_cities, 'n_assets': 300, 'gdp': 2800e9},
    'Jamaïque': {'cities': jamaica_cities, 'n_assets': 150, 'gdp': 15e9},
    'Bangladesh': {'cities': bangladesh_cities, 'n_assets': 250, 'gdp': 460e9},
}

exposures = {}
for country_name, params in countries.items():
    gdf = create_synthetic_exposure(country_name, params['cities'], params['n_assets'], params['gdp'])
    exposures[country_name] = gdf
    print(f"{country_name:12s} : {len(gdf)} actifs, valeur = {gdf['value'].sum()/1e9:.1f} Mds USD")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (country, gdf) in enumerate(exposures.items()):
    ax = axes[idx]
    sc = ax.scatter(gdf.geometry.x, gdf.geometry.y, c=gdf['value'] / 1e9,
                    s=10, cmap='YlOrRd', vmin=0)
    ax.set_title(f'{country}')
    ax.set_xlabel('Longitude')
    if idx == 0:
        ax.set_ylabel('Latitude')
    plt.colorbar(sc, ax=ax, label='Valeur (Mds USD)', shrink=0.8)

plt.suptitle('Exposition synthétique par pays', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 2. Profils multi-péril synthétiques

Pour chaque pays, on génère des aléas synthétiques correspondant à son profil de risque et on calcule l'EAI par péril.

In [ ]:
def create_synthetic_hazard(gdf, haz_type, n_events, max_intensity, spatial_corr=2.0):
    """Créer un aléa synthétique adapté à une exposition."""
    lons = gdf.geometry.x.values
    lats = gdf.geometry.y.values
    
    centroids = Centroids.from_lat_lon(lats, lons)
    frequency = np.ones(n_events) / n_events
    
    intensity_list = []
    for i in range(n_events):
        # Centre aléatoire
        center_idx = np.random.randint(len(lons))
        center_lon, center_lat = lons[center_idx], lats[center_idx]
        
        dist = np.sqrt((lons - center_lon)**2 + (lats - center_lat)**2)
        intensity = max_intensity * np.random.uniform(0.3, 1.0) * np.exp(-0.5 * (dist / spatial_corr)**2)
        intensity += np.random.normal(0, max_intensity * 0.1, len(lons))
        intensity = np.maximum(intensity, 0)
        
        # Seuil
        threshold = max_intensity * 0.1
        intensity[intensity < threshold] = 0
        
        intensity_list.append(csr_matrix(intensity))
    
    haz = Hazard(
        haz_type=haz_type,
        centroids=centroids,
        intensity=vstack(intensity_list),
        frequency=frequency,
        event_id=np.arange(1, n_events + 1),
        event_name=[f'{haz_type}_{i+1:03d}' for i in range(n_events)],
        date=pd.date_range('1980-01-01', periods=n_events, freq='365D').to_numpy().astype('datetime64[ns]').astype(int) // 10**9,
        units='intensity'
    )
    return haz

# Profils de risque par pays
hazard_profiles = {
    'France': {
        'WS': {'n': 30, 'max_int': 45, 'corr': 3.0, 'name': 'Tempête'},
        'RF': {'n': 40, 'max_int': 3.0, 'corr': 1.0, 'name': 'Inondation fluviale'},
        'DR': {'n': 30, 'max_int': 2.5, 'corr': 4.0, 'name': 'Sécheresse'},
        'WF': {'n': 20, 'max_int': 300, 'corr': 0.5, 'name': 'Feux de forêt'},
    },
    'Jamaïque': {
        'TC': {'n': 50, 'max_int': 70, 'corr': 1.5, 'name': 'Cyclone (vent)'},
        'TR': {'n': 50, 'max_int': 200, 'corr': 1.0, 'name': 'Cyclone (pluie)'},
        'CF': {'n': 30, 'max_int': 2.0, 'corr': 0.5, 'name': 'Submersion côtière'},
    },
    'Bangladesh': {
        'TC': {'n': 40, 'max_int': 65, 'corr': 2.0, 'name': 'Cyclone'},
        'RF': {'n': 50, 'max_int': 4.0, 'corr': 3.0, 'name': 'Inondation fluviale'},
        'CF': {'n': 40, 'max_int': 3.0, 'corr': 2.0, 'name': 'Submersion côtière'},
    },
}

# Impact functions simples par type
impact_funcs = {
    'WS': (np.arange(0, 80, 0.5), lambda x: np.clip(((x - 20) / 50)**2, 0, 1) * (x > 20)),
    'RF': (np.arange(0, 5, 0.05), lambda x: np.clip(x / 3, 0, 0.8)),
    'DR': (np.arange(0, 4, 0.01), lambda x: np.clip((x - 0.5) / 2.5, 0, 0.6) * (x > 0.5)),
    'WF': (np.arange(0, 800, 1), lambda x: 1 / (1 + (295 / np.maximum(x, 1))**2.5) * (x > 10)),
    'TC': (np.arange(0, 100, 0.5), lambda x: np.clip(((x - 25) / 50)**3, 0, 1) * (x > 25)),
    'TR': (np.arange(0, 400, 1), lambda x: np.clip(x / 300, 0, 0.6)),
    'CF': (np.arange(0, 5, 0.05), lambda x: np.clip(x / 2, 0, 0.9)),
}

# Calcul EAI pour chaque pays × péril
all_results = {}

for country_name, profile in hazard_profiles.items():
    gdf = exposures[country_name].copy()
    
    # Ajouter les colonnes impf pour chaque type
    for haz_type in profile:
        gdf[f'impf_{haz_type}'] = 1
    
    exp = Exposures(gdf)
    exp.check()
    
    country_results = {}
    for haz_type, params in profile.items():
        haz = create_synthetic_hazard(gdf, haz_type, params['n'], params['max_int'], params['corr'])
        
        intensity_range, mdr_func = impact_funcs[haz_type]
        mdr = mdr_func(intensity_range)
        
        impf = ImpactFunc(id=1, haz_type=haz_type, intensity=intensity_range,
                          mdd=mdr, paa=np.ones_like(mdr), intensity_unit='intensity',
                          name=params['name'])
        impf_set = ImpactFuncSet([impf])
        
        imp = Impact()
        imp.calc(exp, impf_set, haz, save_mat=False)
        
        country_results[haz_type] = {
            'eai': imp.aai_agg,
            'name': params['name'],
            'max_event': imp.at_event.max()
        }
    
    all_results[country_name] = country_results
    
    print(f"\n=== {country_name} ===")
    total_eai = 0
    for haz_type, res in country_results.items():
        print(f"  {res['name']:25s} — EAI: {res['eai']/1e9:>8.3f} Mds USD")
        total_eai += res['eai']
    print(f"  {'TOTAL':25s} — EAI: {total_eai/1e9:>8.3f} Mds USD")
    gdp = countries[country_name]['gdp']
    print(f"  {'EAI/GDP':25s} —      {total_eai/gdp*100:>8.3f} %")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

colors_map = {
    'WS': '#2196F3', 'RF': '#4CAF50', 'DR': '#FF9800', 'WF': '#F44336',
    'TC': '#9C27B0', 'TR': '#00BCD4', 'CF': '#795548'
}

for idx, (country, results) in enumerate(all_results.items()):
    ax = axes[idx]
    
    names = [r['name'] for r in results.values()]
    eais = [r['eai'] / 1e6 for r in results.values()]
    haz_types = list(results.keys())
    colors = [colors_map.get(ht, 'gray') for ht in haz_types]
    
    bars = ax.bar(names, eais, color=colors, alpha=0.8)
    ax.set_ylabel('EAI (M USD/an)')
    ax.set_title(f'{country}')
    ax.tick_params(axis='x', rotation=30)
    
    # Annoter
    for bar, val in zip(bars, eais):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                    f'{val:.0f}M', ha='center', va='bottom', fontsize=8)

plt.suptitle('Profil de risque multi-péril par pays', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# EAI/GDP par pays
country_names = list(all_results.keys())
gdps = [countries[c]['gdp'] for c in country_names]
total_eais = [sum(r['eai'] for r in all_results[c].values()) for c in country_names]
eai_gdp_pct = [e / g * 100 for e, g in zip(total_eais, gdps)]

bars = axes[0].bar(country_names, eai_gdp_pct, color=['#2196F3', '#FF9800', '#4CAF50'], alpha=0.8)
axes[0].set_ylabel('EAI / GDP (%)')
axes[0].set_title('Exposition relative au risque climatique')
for bar, val in zip(bars, eai_gdp_pct):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                 f'{val:.2f}%', ha='center', va='bottom', fontsize=10)

# Stacked bar : contribution par péril
bottom = np.zeros(len(country_names))
for haz_type in set(ht for c in all_results.values() for ht in c):
    vals = []
    for c in country_names:
        if haz_type in all_results[c]:
            vals.append(all_results[c][haz_type]['eai'] / 1e9)
        else:
            vals.append(0)
    
    name = all_results[list(all_results.keys())[0]].get(haz_type, {}).get('name', haz_type)
    for c_idx, c in enumerate(country_names):
        if haz_type in all_results[c]:
            name = all_results[c][haz_type]['name']
            break
    
    axes[1].bar(country_names, vals, bottom=bottom, 
                color=colors_map.get(haz_type, 'gray'), alpha=0.8, label=name)
    bottom += vals

axes[1].set_ylabel('EAI (Mds USD/an)')
axes[1].set_title('EAI par péril et par pays')
axes[1].legend(loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()

## 3. Analyse coût-bénéfice — Mesures d'adaptation

Le moteur `CostBenefit` de CLIMADA évalue la rentabilité des mesures d'adaptation :

$$NPV = \sum_{t=0}^{T} \frac{B_t - C_t}{(1+r)^t}$$

Où :
- $B_t$ = bénéfice (réduction des pertes) à l'année $t$
- $C_t$ = coût de la mesure à l'année $t$
- $r$ = taux d'actualisation
- $T$ = horizon temporel

In [ ]:
# Simulation coût-bénéfice simplifiée pour la France
# Mesures d'adaptation avec leurs caractéristiques

adaptation_measures = {
    'Digues fluviales': {
        'target_hazard': 'RF',
        'risk_reduction': 0.60,  # 60% de réduction du risque
        'cost_initial': 5e9,     # 5 Mds EUR investissement
        'cost_annual': 100e6,    # 100 M EUR maintenance/an
        'lifetime': 50,          # durée de vie 50 ans
    },
    'Normes construction tempête': {
        'target_hazard': 'WS',
        'risk_reduction': 0.40,
        'cost_initial': 2e9,
        'cost_annual': 50e6,
        'lifetime': 30,
    },
    'Gestion forêt (débroussaillage)': {
        'target_hazard': 'WF',
        'risk_reduction': 0.50,
        'cost_initial': 500e6,
        'cost_annual': 200e6,
        'lifetime': 20,
    },
    'Irrigation efficiente': {
        'target_hazard': 'DR',
        'risk_reduction': 0.35,
        'cost_initial': 3e9,
        'cost_annual': 150e6,
        'lifetime': 25,
    },
    'Système alerte précoce': {
        'target_hazard': 'all',
        'risk_reduction': 0.15,  # réduction modeste mais transversale
        'cost_initial': 200e6,
        'cost_annual': 30e6,
        'lifetime': 15,
    },
}

# Calcul NPV pour chaque mesure
discount_rate = 0.03  # 3%
france_results = all_results['France']

npv_results = {}
for measure_name, params in adaptation_measures.items():
    T = params['lifetime']
    
    # Bénéfice annuel = réduction du risque × EAI concerné
    if params['target_hazard'] == 'all':
        eai_target = sum(r['eai'] for r in france_results.values())
    else:
        eai_target = france_results.get(params['target_hazard'], {}).get('eai', 0)
    
    annual_benefit = eai_target * params['risk_reduction']
    
    # NPV
    npv = -params['cost_initial']  # investissement initial
    cumul_benefit = 0
    cumul_cost = params['cost_initial']
    
    for t in range(1, T + 1):
        discount = 1 / (1 + discount_rate)**t
        npv += (annual_benefit - params['cost_annual']) * discount
        cumul_benefit += annual_benefit
        cumul_cost += params['cost_annual']
    
    bcr = cumul_benefit / cumul_cost  # Benefit-Cost Ratio
    payback = None
    running_npv = -params['cost_initial']
    for t in range(1, T + 1):
        running_npv += (annual_benefit - params['cost_annual']) / (1 + discount_rate)**t
        if running_npv > 0 and payback is None:
            payback = t
    
    npv_results[measure_name] = {
        'npv': npv,
        'bcr': bcr,
        'payback': payback,
        'annual_benefit': annual_benefit,
        'total_cost': cumul_cost,
        'lifetime': T,
    }
    
    print(f"\n--- {measure_name} ---")
    print(f"  Bénéfice annuel :  {annual_benefit/1e6:>8.1f} M EUR")
    print(f"  Coût total :       {cumul_cost/1e9:>8.2f} Mds EUR")
    print(f"  NPV :              {npv/1e9:>8.2f} Mds EUR")
    print(f"  BCR :              {bcr:>8.2f}")
    print(f"  Retour investt :   {payback if payback else '>'+str(T)} ans")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

measure_names = list(npv_results.keys())
short_names = ['Digues', 'Normes\ntempête', 'Gestion\nforêt', 'Irrigation', 'Alerte\nprécoce']

# 1. NPV
npvs = [npv_results[m]['npv'] / 1e9 for m in measure_names]
colors = ['green' if v > 0 else 'red' for v in npvs]
axes[0, 0].bar(short_names, npvs, color=colors, alpha=0.8)
axes[0, 0].axhline(y=0, color='black', linewidth=0.5)
axes[0, 0].set_ylabel('NPV (Mds EUR)')
axes[0, 0].set_title('Valeur Actuelle Nette')

# 2. BCR
bcrs = [npv_results[m]['bcr'] for m in measure_names]
colors_bcr = ['green' if v > 1 else 'red' for v in bcrs]
axes[0, 1].bar(short_names, bcrs, color=colors_bcr, alpha=0.8)
axes[0, 1].axhline(y=1.0, color='black', linestyle='--', label='Seuil rentabilité')
axes[0, 1].set_ylabel('BCR (Benefit-Cost Ratio)')
axes[0, 1].set_title('Ratio bénéfice-coût')
axes[0, 1].legend()

# 3. Payback period
paybacks = [npv_results[m]['payback'] or npv_results[m]['lifetime'] for m in measure_names]
lifetimes = [npv_results[m]['lifetime'] for m in measure_names]
x = np.arange(len(measure_names))
axes[1, 0].bar(x - 0.2, paybacks, 0.35, label='Retour investissement', color='steelblue', alpha=0.8)
axes[1, 0].bar(x + 0.2, lifetimes, 0.35, label='Durée de vie', color='lightblue', alpha=0.8)
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(short_names, fontsize=9)
axes[1, 0].set_ylabel('Années')
axes[1, 0].set_title('Payback vs Durée de vie')
axes[1, 0].legend()

# 4. Bénéfice annuel vs coût annualisé
annual_benefits = [npv_results[m]['annual_benefit'] / 1e6 for m in measure_names]
annual_costs = [npv_results[m]['total_cost'] / npv_results[m]['lifetime'] / 1e6 for m in measure_names]
axes[1, 1].scatter(annual_costs, annual_benefits, s=200, c=bcrs, cmap='RdYlGn', 
                    vmin=0, vmax=max(bcrs) * 1.2, edgecolors='black', zorder=3)
for i, name in enumerate(short_names):
    axes[1, 1].annotate(name, (annual_costs[i], annual_benefits[i]),
                         textcoords="offset points", xytext=(10, 5), fontsize=8)
# Ligne BCR=1
max_val = max(max(annual_costs), max(annual_benefits))
axes[1, 1].plot([0, max_val], [0, max_val], 'k--', alpha=0.3, label='BCR = 1')
axes[1, 1].set_xlabel('Coût annualisé (M EUR/an)')
axes[1, 1].set_ylabel('Bénéfice annuel (M EUR/an)')
axes[1, 1].set_title('Efficience des mesures')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 4. Roadmap d'adaptation — Priorisation

Classement des mesures par efficience (BCR) et urgence (risque résiduel).

In [ ]:
# Tableau récapitulatif
roadmap_df = pd.DataFrame({
    'Mesure': measure_names,
    'NPV (Mds EUR)': [npv_results[m]['npv'] / 1e9 for m in measure_names],
    'BCR': [npv_results[m]['bcr'] for m in measure_names],
    'Payback (ans)': [npv_results[m]['payback'] or 'N/A' for m in measure_names],
    'Bénéfice annuel (M EUR)': [npv_results[m]['annual_benefit'] / 1e6 for m in measure_names],
    'Priorité': [''] * len(measure_names)
})

# Priorisation simple basée sur BCR
for i in range(len(roadmap_df)):
    bcr = roadmap_df.loc[i, 'BCR']
    if bcr > 2:
        roadmap_df.loc[i, 'Priorité'] = 'Haute'
    elif bcr > 1:
        roadmap_df.loc[i, 'Priorité'] = 'Moyenne'
    else:
        roadmap_df.loc[i, 'Priorité'] = 'Basse'

roadmap_df = roadmap_df.sort_values('BCR', ascending=False)
print("=== ROADMAP D'ADAPTATION — France ===\n")
print(roadmap_df.to_string(index=False))

# Risque résiduel après implémentation de toutes les mesures
print("\n\n=== RISQUE RÉSIDUEL ===")
total_eai_fra = sum(r['eai'] for r in france_results.values())
residual_eai = total_eai_fra

for measure_name, params in adaptation_measures.items():
    if params['target_hazard'] == 'all':
        reduction = residual_eai * params['risk_reduction']
    elif params['target_hazard'] in france_results:
        reduction = france_results[params['target_hazard']]['eai'] * params['risk_reduction']
    else:
        reduction = 0
    residual_eai -= reduction

print(f"EAI initial :  {total_eai_fra/1e9:.3f} Mds EUR")
print(f"EAI résiduel : {residual_eai/1e9:.3f} Mds EUR")
print(f"Réduction :    {(1 - residual_eai/total_eai_fra)*100:.1f}%")

## 5. Comparaison internationale

Synthèse des profils de risque et des besoins d'adaptation.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Bubble chart : GDP vs EAI vs EAI/GDP
for idx, (country, results) in enumerate(all_results.items()):
    gdp = countries[country]['gdp'] / 1e9
    total_eai = sum(r['eai'] for r in results.values()) / 1e6
    eai_gdp = total_eai / (gdp * 1000) * 100  # %
    
    # Taille = EAI/GDP (vulnérabilité relative)
    size = eai_gdp * 500
    
    ax.scatter(gdp, total_eai, s=size, alpha=0.6, zorder=3, 
              label=f'{country} (EAI/GDP = {eai_gdp:.2f}%)')
    ax.annotate(country, (gdp, total_eai), textcoords="offset points",
                xytext=(15, 5), fontsize=12, fontweight='bold')

ax.set_xlabel('GDP (Mds USD)', fontsize=12)
ax.set_ylabel('EAI total (M USD/an)', fontsize=12)
ax.set_title('Profil de risque climatique — Comparaison internationale', fontsize=14)
ax.set_xscale('log')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Synthèse

### Profils de risque

| Pays | PIB | EAI total | EAI/PIB | Péril dominant |
|------|-----|-----------|---------|----------------|
| France | 2 800 Mds | Variable | ~0.1-0.3% | Tempêtes, inondations |
| Jamaïque | 15 Mds | Variable | ~1-3% | Cyclones tropicaux |
| Bangladesh | 460 Mds | Variable | ~2-5% | Inondations, cyclones |

### Points clés
- **France** : risque diversifié mais modéré en % du PIB — adaptation rentable sur les digues et la gestion forestière
- **Jamaïque** : très exposée aux cyclones — l'EAI/PIB élevé justifie des investissements massifs en résilience
- **Bangladesh** : cumul inondation + cyclone — le pays le plus vulnérable en termes relatifs
- Les systèmes d'alerte précoce sont la mesure la plus universellement rentable (BCR élevé, faible coût)

### Outils CLIMADA utilisés
| Outil | Usage |
|-------|-------|
| `LitPop` / `BlackMarble` | Exposition nationale haute résolution |
| `Impact.calc()` | Calcul des pertes par péril |
| `CostBenefit` | Analyse coût-bénéfice des mesures |
| `UQ Engine` | Quantification d'incertitude (Monte Carlo) |

### Prochaines étapes
- Charger des expositions LitPop réelles à haute résolution
- Utiliser les aléas des notebooks précédents (01-10) comme input
- Lancer le moteur CostBenefit de CLIMADA avec des mesures CLIMADA natives
- Explorer l'UQ engine pour quantifier l'incertitude des résultats
- Comparer avec les National Adaptation Plans (NAP) existants